# Fish Speech S2 - Interactive Inference

RTX 3090 24GB，meta device 加载，完整 32768 上下文。

## 0. 环境初始化

In [ ]:
import os, sys, torch, numpy as np, time
from pathlib import Path
from IPython.display import Audio, display

os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["PYTHONUTF8"] = "1"
sys.stdout.reconfigure(encoding="utf-8", errors="replace")
sys.stderr.reconfigure(encoding="utf-8", errors="replace")

# ===== 配置 =====
CHECKPOINT = Path("checkpoints/s2-pro")
CODEC_PATH = CHECKPOINT / "codec.pth"
DEVICE = "cuda"
PRECISION = torch.bfloat16  # RTX 3090 用 bf16，不要用 fp16
MAX_SEQ_LEN = 0              # 0 = 用模型默认值 (32768)，设小可省显存提速

print(f"PyTorch {torch.__version__}")
print(f"CUDA: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

## 1. 加载模型（LLM + Codec）

In [ ]:
from fish_speech.models.text2semantic.inference import (
    init_model, load_codec_model, encode_audio, decode_to_audio, generate_long,
)
from fish_speech.models.text2semantic.llama import precompute_freqs_cis
import soundfile as sf

t0 = time.time()

# LLM（meta device init，直接加载到 GPU）
model, decode_one_token = init_model(CHECKPOINT, DEVICE, PRECISION, compile=False)

# 重建 buffer（meta device init 后是 placeholder）
max_seq = min(model.config.max_seq_len, MAX_SEQ_LEN) if MAX_SEQ_LEN > 0 else model.config.max_seq_len
model.config.max_seq_len = max_seq
model.register_buffer("freqs_cis",
    precompute_freqs_cis(max_seq, model.config.head_dim, model.config.rope_base).to(DEVICE),
    persistent=False)
model.register_buffer("causal_mask",
    torch.tril(torch.ones(max_seq, max_seq, dtype=torch.bool, device=DEVICE)),
    persistent=False)
with torch.device(DEVICE):
    model.setup_caches(max_batch_size=1, max_seq_len=max_seq, dtype=PRECISION)

# Codec
codec = load_codec_model(CODEC_PATH, DEVICE, PRECISION)

torch.cuda.synchronize()
print(f"加载完成: {time.time()-t0:.1f}s | max_seq_len={max_seq} | VRAM: {torch.cuda.memory_allocated()/1e9:.2f} GB")

In [5]:
def tts(text, seed=42, temperature=0.7, top_p=0.9, top_k=30,
        prompt_text=None, prompt_tokens=None, out_path=None):
    """生成语音并播放。prompt_text/prompt_tokens 用于 voice clone。"""
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    codes = []
    for r in generate_long(
        model=model, device=DEVICE, decode_one_token=decode_one_token,
        text=text, num_samples=1, max_new_tokens=0,
        top_p=top_p, top_k=top_k, temperature=temperature,
        compile=False, iterative_prompt=True, chunk_length=300,
        prompt_text=prompt_text, prompt_tokens=prompt_tokens,
    ):
        if r.action == "sample":
            codes.append(r.codes)
        elif r.action == "next":
            break
    if not codes:
        print("未生成任何 token"); return None
    merged = torch.cat(codes, dim=1)
    audio = decode_to_audio(merged.to(DEVICE), codec)
    wav = audio.cpu().float().numpy()
    if out_path:
        sf.write(out_path, wav, codec.sample_rate)
        print(f"已保存: {out_path} ({len(wav)/codec.sample_rate:.2f}s)")
    display(Audio(data=wav, rate=codec.sample_rate))
    return wav

## 2. 随机音色生成（无参考音频）

In [6]:
tts("<|speaker:0|>你好，欢迎使用语音合成系统，今天天气真不错。", out_path="output_notebook.wav")

2026-03-17 14:10:43.765 | INFO     | fish_speech.models.text2semantic.inference:generate_long:668 - Split into 1 turns, grouped into 1 batches
2026-03-17 14:10:43.765 | INFO     | fish_speech.models.text2semantic.inference:generate_long:680 - --- Sample 0, Batch 0 (79 bytes) ---
2026-03-17 14:10:43.765 | INFO     | fish_speech.models.text2semantic.inference:generate_long:684 - Batch text: <|speaker:0|>你好，欢迎使用语音合成系统，今天天气真不错。
2026-03-17 14:10:43.766 | INFO     | fish_speech.models.text2semantic.inference:generate_long:710 - Visualizing prompt structure:
2026-03-17 14:10:43.771 | INFO     | fish_speech.models.text2semantic.inference:generate_long:721 - Encoded prompt shape: torch.Size([11, 40])


<|im_start|>system
convert the provided text to speech<|im_end|>
<|im_start|>user
<|speaker:0|>你好，欢迎使用语音合成系统，今天天气真不错。<|im_end|>
<|im_start|>assistant
<|voice|>


  0%|          | 97/29959 [00:27<2:21:16,  3.52it/s]
2026-03-17 14:11:12.186 | INFO     | fish_speech.models.text2semantic.inference:generate_long:758 - Batch 0: Generated 99 tokens in 28.42 seconds, 3.48 tokens/sec
2026-03-17 14:11:12.186 | INFO     | fish_speech.models.text2semantic.inference:generate_long:762 - Bandwidth achieved: 15.89 GB/s
2026-03-17 14:11:12.188 | INFO     | fish_speech.models.text2semantic.inference:generate_long:788 - GPU Memory used: 20.75 GB


已保存: output_notebook.wav (4.55s)


array([-0.00033569, -0.00057983, -0.00039673, ..., -0.00021362,
        0.        , -0.00027466], dtype=float32)

## 3. Voice Clone（参考音频克隆）

In [7]:
ref_audio = r"C:\QBS\AI_tools\models\GPT-SoVITS v2 pro plus\诗歌剧\语音\町内会の秋祭り、行きませんか？極秘情報なんですが…なんと、お団子食べ放題らしいですっ！.mp3"
ref_text = "町内会の秋祭り、行きませんか？極秘情報なんですが…なんと、お団子食べ放題らしいですっ！"

# 编码参考音频（只需跑一次，换参考音频时重新跑）
ref_codes = encode_audio(ref_audio, codec, DEVICE)
print(f"参考音频 VQ codes: {ref_codes.shape}")

c:\QBS\AI_tools\fish-speech\.venv\Lib\site-packages\torchaudio\_backend\utils.py:213: UserWarning: In 2.9, this function's implementation will be changed to use torchaudio.load_with_torchcodec` under the hood. Some parameters like ``normalize``, ``format``, ``buffer_size``, and ``backend`` will be ignored. We recommend that you port your code to rely directly on TorchCodec's decoder instead: https://docs.pytorch.org/torchcodec/stable/generated/torchcodec.decoders.AudioDecoder.html#torchcodec.decoders.AudioDecoder.
  warnings.warn(


参考音频 VQ codes: torch.Size([10, 246])


In [9]:
tts("最后，哥哥——还是带着我出门了。我无声地感谢着哥哥——的帮忙。哥哥——加快了脚步，我被晃得更厉害了。走出家门后，哥哥——停下脚步。",
    prompt_text=[ref_text], prompt_tokens=[ref_codes.cpu()],
    out_path="output_clone_notebook.wav")

2026-03-17 14:17:10.115 | INFO     | fish_speech.models.text2semantic.inference:generate_long:668 - Split into 0 turns, grouped into 1 batches
2026-03-17 14:17:10.116 | INFO     | fish_speech.models.text2semantic.inference:generate_long:680 - --- Sample 0, Batch 0 (195 bytes) ---
2026-03-17 14:17:10.116 | INFO     | fish_speech.models.text2semantic.inference:generate_long:684 - Batch text: 最后，哥哥——还是带着我出门了。我无声地感谢着哥哥——的帮忙。哥哥——加快了脚步，我被晃得更厉害了。走出家门后，哥哥——停下脚步。
2026-03-17 14:17:10.116 | INFO     | fish_speech.models.text2semantic.inference:generate_long:710 - Visualizing prompt structure:
2026-03-17 14:17:10.122 | INFO     | fish_speech.models.text2semantic.inference:generate_long:721 - Encoded prompt shape: torch.Size([11, 358])


<|im_start|>system
convert the provided text to speech reference to the following:

Text:
<|speaker:0|>町内会の秋祭り、行きませんか？極秘情報なんですが…なんと、お��子食べ放題らしいですっ！

Speech:
[<|semantic|>x246]<|im_end|>
<|im_start|>user
最后，哥哥——还是带着我出门了。我无声地感谢着哥哥——的帮忙。哥哥——加快了脚步，我被晃得更厉害了。走出家门后，哥哥——停下脚步。<|im_end|>
<|im_start|>assistant
<|voice|>


  1%|          | 288/29641 [01:23<2:21:21,  3.46it/s]
2026-03-17 14:18:34.080 | INFO     | fish_speech.models.text2semantic.inference:generate_long:758 - Batch 0: Generated 290 tokens in 83.97 seconds, 3.45 tokens/sec
2026-03-17 14:18:34.081 | INFO     | fish_speech.models.text2semantic.inference:generate_long:762 - Bandwidth achieved: 15.76 GB/s
2026-03-17 14:18:34.081 | INFO     | fish_speech.models.text2semantic.inference:generate_long:788 - GPU Memory used: 20.76 GB


已保存: output_clone_notebook.wav (13.42s)


array([ 2.2888184e-03,  8.2397461e-04,  1.3732910e-03, ...,
        3.0517578e-05, -4.5776367e-04,  3.0517578e-05], dtype=float32)

## 4. 多说话人 / 情感控制

In [20]:
tts("""<|speaker:0|>你好啊，今天过得怎么样？
<|speaker:1|>[super happy]太棒了！我刚收到了一个好消息！""",
    seed=123, temperature=0.8, out_path="output_multispeaker.wav")

2026-03-17 11:38:32.541 | INFO     | fish_speech.models.text2semantic.inference:generate_long:668 - Split into 2 turns, grouped into 1 batches
2026-03-17 11:38:32.541 | INFO     | fish_speech.models.text2semantic.inference:generate_long:680 - --- Sample 0, Batch 0 (121 bytes) ---
2026-03-17 11:38:32.542 | INFO     | fish_speech.models.text2semantic.inference:generate_long:684 - Batch text: <|speaker:0|>你好啊，今天过得怎么样？
<|speaker:1|>[super happy]太棒了！我刚收到了一个好消息！
2026-03-17 11:38:32.542 | INFO     | fish_speech.models.text2semantic.inference:generate_long:710 - Visualizing prompt structure:
2026-03-17 11:38:32.545 | INFO     | fish_speech.models.text2semantic.inference:generate_long:721 - Encoded prompt shape: torch.Size([11, 54])


<|im_start|>system
convert the provided text to speech<|im_end|>
<|im_start|>user
<|speaker:0|>你好啊，今天过得怎么样？
<|speaker:1|>[super happy]太棒了！我刚收到了一个好消息！<|im_end|>
<|im_start|>assistant
<|voice|>


  1%|          | 113/16329 [00:21<51:52,  5.21it/s]
2026-03-17 11:38:54.382 | INFO     | fish_speech.models.text2semantic.inference:generate_long:758 - Batch 0: Generated 115 tokens in 21.84 seconds, 5.27 tokens/sec
2026-03-17 11:38:54.382 | INFO     | fish_speech.models.text2semantic.inference:generate_long:762 - Bandwidth achieved: 24.02 GB/s
2026-03-17 11:38:54.383 | INFO     | fish_speech.models.text2semantic.inference:generate_long:788 - GPU Memory used: 17.13 GB


已保存: output_multispeaker.wav (5.29s)


array([ 5.0354004e-03,  4.1198730e-03,  3.8757324e-03, ...,
        0.0000000e+00, -3.0517578e-05, -1.8310547e-04], dtype=float32)

## 5. 工具

In [17]:
# 显存状态
if torch.cuda.is_available():
    print(f"显存已分配: {torch.cuda.memory_allocated()/1e9:.2f} GB")
    print(f"显存已缓存: {torch.cuda.memory_reserved()/1e9:.2f} GB")
    print(f"显存峰值:   {torch.cuda.max_memory_allocated()/1e9:.2f} GB")

显存已分配: 18.66 GB
显存已缓存: 18.78 GB
显存峰值:   20.63 GB


In [ ]:
# 清理显存（需要重新运行 cell 1 加载模型）
# del model, codec, decode_one_token
torch.cuda.empty_cache()
print("已清理")

已清理


: 